## 1. Check A100 accelerator

In [ ]:
import torch, subprocess, sys
assert torch.cuda.is_available(), "CUDA not available. Enable an A100 GPU runtime in Colab."
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu, "VRAM GiB:", round(vram, 1))
subprocess.run(["nvidia-smi"], check=True)
if "A100" not in gpu:
    raise RuntimeError(f"Expected an A100, got {gpu}. Change the runtime or deliberately update this guard.")

## 2. Mount Drive and clone the exact source

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, shutil, subprocess, time, hashlib

REPO_URL = "https://github.com/utsab818/dpr-mining-study.git"
REPO = Path('/content/dpr-mining-study')
DRIVE_ROOT = Path('/content/drive/MyDrive/dpr_v3_a100_seed42')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Git commit:', commit)
prior_commit_file = DRIVE_ROOT / 'source_commit.txt'
if prior_commit_file.exists():
    prior_commit = prior_commit_file.read_text().strip()
    if prior_commit != commit:
        raise RuntimeError(
            f'Drive root belongs to commit {prior_commit}, but cloned {commit}. '
            'Use a new DRIVE_ROOT for the new revision.'
        )
for name in ['data.py','model.py','loss.py','mine_negatives.py','train.py','evaluate.py','warmup_v3.py']:
    assert (REPO / 'src' / name).exists(), f'Missing src/{name} in GitHub. Push latest V3 code first.'
print('All expected source files found.')

## 3. Install dependencies and record environment

In [ ]:
# Keep the Colab-provided CUDA-compatible torch build.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.35,<5', 'datasets>=2.16,<5', 'safetensors'], check=True)
import transformers, datasets
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)
print('cuda:', torch.version.cuda)
(DRIVE_ROOT / 'environment.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
(DRIVE_ROOT / 'source_commit.txt').write_text(commit + '\n')

## 4. Reusable stage runner with Drive-backed outputs

In [ ]:
import shlex

def run_stage(name, args, artifacts):
    """Restore a complete stage or run it and copy outputs to Drive.

    args are passed directly to Python (no shell interpretation).
    artifacts are paths relative to the repository.
    """
    stage_dir = DRIVE_ROOT / name
    stage_dir.mkdir(parents=True, exist_ok=True)
    local_paths = [REPO / p for p in artifacts]
    saved_paths = [stage_dir / p for p in artifacts]
    if all(p.is_file() for p in saved_paths):
        for dest, src in zip(local_paths, saved_paths):
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.is_file(): shutil.copy2(src, dest)
        print(f'RESTORED / SKIPPED completed stage: {name}')
        return
    if any(p.is_file() for p in saved_paths):
        raise RuntimeError(f'Partial Drive output for {name}; inspect {stage_dir} before retrying.')
    for p in local_paths:
        if p.exists():
            raise FileExistsError(f'Local output exists without recorded Drive stage: {p}')
    print('RUNNING:', name, shlex.join(args), flush=True)
    t0 = time.perf_counter()
    with (stage_dir / 'stdout.log').open('w') as log:
        proc = subprocess.Popen([sys.executable, '-u', '-m', *args],
                                cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        rc = proc.wait()
    if rc != 0: raise RuntimeError(f'{name} failed with return code {rc}; see Drive stdout.log')
    for local, dest in zip(local_paths, saved_paths):
        if not local.is_file(): raise FileNotFoundError(f'Missing output: {local}')
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local, dest)
    (stage_dir / 'completed.json').write_text(json.dumps({
        'name': name, 'git_commit': commit, 'seconds': round(time.perf_counter() - t0, 2),
        'outputs': artifacts}, indent=2))
    print('SAVED completed stage to Drive:', stage_dir)

# A new Drive directory should be used for each new code revision or experimental run.

## 5. Train the V3 warm-up retriever

In [ ]:
WARM_DIR = 'experiments/v3/warmup/full'
WARM_BASE = 'dpr_warmup_n2000_b32_seed42_steps62'
WARM_PT = f'{WARM_DIR}/{WARM_BASE}.pt'
run_stage('01_warmup', ['src.warmup_v3', '--max-samples','2000',
  '--batch-size','32','--epochs','1','--seed','42','--output-dir',WARM_DIR],
  [f'{WARM_DIR}/{WARM_BASE}{suffix}' for suffix in ('.pt','.json','_steps.csv')])

## 6. Mine model-hard negatives from 5,000 passages

In [ ]:
MINE_DIR = 'experiments/v3/mining/full'
MINE_PT = f'{MINE_DIR}/mined_negatives.pt'
run_stage('02_mining', ['src.mine_negatives','--num-questions','2000',
  '--num-passages','5000','--num-hard','4','--batch-size','32',
  '--seed','42','--checkpoint',WARM_PT,'--output-dir',MINE_DIR],
  [MINE_PT, f'{MINE_DIR}/passage_index.pt', f'{MINE_DIR}/manifest.json'])

### Mining sanity check
This checks positive exclusion and unique IDs. Exact answer overlap was tested in the local smoke run; it remains a heuristic and cannot rule out semantically relevant false negatives.

In [ ]:
mined = torch.load(REPO / MINE_PT, map_location='cpu', weights_only=True)
assert len(mined['questions']) == 2000
assert len(mined['passages']) == 5000
hard_ids = mined['hard_negative_indices']
assert tuple(hard_ids.shape) == (2000,4)
assert bool(((hard_ids >= 0) & (hard_ids < 5000)).all())
for i in range(2000):
    row = hard_ids[i].tolist()
    assert i not in row and len(set(row)) == 4
print('Mining ID sanity checks passed.')

## 7. Four second-stage strategies with identical question batches

In [ ]:
TRAIN_DIR = 'experiments/v3/training/full'
STRATEGIES = ['random','hard','mixed','inbatch']
CHECKPOINTS = {}
for strategy in STRATEGIES:
    k = 32 if strategy == 'inbatch' else 4
    base = f'dpr_{strategy}_k{k}_b33_seed42_steps60'
    args = ['src.train','--strategy',strategy,'--mined-path',MINE_PT,
      '--warmup-path',WARM_PT,'--output-dir',TRAIN_DIR,
      '--batch-size','33','--num-negatives',str(k),
      '--seed','42','--max-steps','60','--log-every','10']
    run_stage('03_train_' + strategy, args,
      [f'{TRAIN_DIR}/{base}{suffix}' for suffix in ('.pt','.json','_steps.csv')])
    CHECKPOINTS[strategy] = f'{TRAIN_DIR}/{base}.pt'
print('All strategy checkpoints:', CHECKPOINTS)

**Fairness note:** Same second-stage seed, 33 query examples per step, and 60 optimizer steps mean the models see matched query batches. Their passage encoder workloads and training costs differ. `mixed` also has more total negatives per query than the other strategies; it is not a quality-only comparison.

## 8. Evaluate held-out retrieval performance

In [ ]:
EVAL_DIR = 'experiments/v3/evaluation/full'
EVAL_OUT = f'{EVAL_DIR}/seed42_1000.csv'
ckpts = [f'warmup={WARM_PT}'] + [f'{s}={p}' for s,p in CHECKPOINTS.items()]
args = ['src.evaluate', *sum((['--checkpoint', spec] for spec in ckpts), []),
  '--max-samples','1000','--batch-size','16','--save-query-ranks',
  '--output',EVAL_OUT]
run_stage('04_evaluation', args, [EVAL_OUT,
    f'{EVAL_DIR}/seed42_1000_query_ranks.csv',
    f'{EVAL_DIR}/seed42_1000_metadata.json'])
import pandas as pd
display(pd.read_csv(REPO / EVAL_OUT))

## 9. Share results

In [ ]:
print('All completed artifacts are in:', DRIVE_ROOT)
print('Share at minimum:')
print('  -', DRIVE_ROOT / '04_evaluation' / EVAL_OUT)
print('  -', DRIVE_ROOT / '04_evaluation' / f'{EVAL_DIR}/seed42_1000_query_ranks.csv')
print('  -', DRIVE_ROOT / '04_evaluation' / f'{EVAL_DIR}/seed42_1000_metadata.json')
print('  - all four 03_train_* metrics JSONs and stdout logs')
print('  - source_commit.txt and environment.txt')
print('For independent verification, keep warm-up, mining artifacts and model checkpoints on Drive.')